# Week 3 Exercise — Synthetic Data Generator

Generate diverse, high-quality training datasets for AI/ML using two models:
- **DeepSeek Chat** — via the DeepSeek API
- **Llama 3.2** — via Ollama (running locally)

### Supported Dataset Types
| Type | Use Case |
|---|---|
| Q&A Pairs | Fine-tuning / evaluation datasets |
| Customer Support Conversations | Chatbot training with intent + sentiment labels |
| Text Classification | Labelled text for classification models |
| Instruction-Response (Alpaca) | Instruction-following fine-tuning |
| Sentiment Analysis | Sentiment labels with reasoning |

### Features
- Switch between models to compare output diversity and quality
- Control the number of examples generated
- View results in a table
- Download as **JSON** or **CSV**

In [ ]:
import os
import re
import json
import csv
import tempfile
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import pandas as pd

load_dotenv(override=True)

# ── Model clients ─────────────────────────────────────────────────────────────
MODELS = {
    "DeepSeek Chat": {
        "client": OpenAI(
            base_url="https://api.deepseek.com/v1",
            api_key=os.getenv("DEEPSEEK_API_KEY")
        ),
        "model": "deepseek-chat"
    },
    "Llama 3.2 (Ollama)": {
        "client": OpenAI(
            base_url="http://localhost:11434/v1",
            api_key="ollama"
        ),
        "model": "llama3.2"
    }
}

# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a synthetic data generation expert for AI/ML training datasets.
Your output must be a valid JSON array and NOTHING ELSE.
No markdown, no code blocks, no explanations — just the raw JSON array starting with [ and ending with ]."""

# ── Dataset type configurations ───────────────────────────────────────────────
DATASET_CONFIGS = {
    "Q&A Pairs": {
        "description": "Question and answer pairs for fine-tuning or evaluation datasets.",
        "columns": ["question", "answer", "difficulty"],
        "prompt": lambda domain, n: (
            f"Generate {n} diverse Q&A pairs about: {domain}\n\n"
            f"Return a JSON array with exactly {n} objects:\n"
            '[\n  {"question": "...", "answer": "...", "difficulty": "easy"}\n]\n\n'
            "difficulty must be one of: easy, medium, hard\n"
            "Make questions varied in style. Answers should be detailed and accurate.\n"
            "Return ONLY the JSON array."
        )
    },
    "Customer Support Conversations": {
        "description": "Support dialogues labelled with customer intent and sentiment.",
        "columns": ["customer_message", "agent_response", "intent", "sentiment"],
        "prompt": lambda domain, n: (
            f"Generate {n} customer support conversation examples about: {domain}\n\n"
            f"Return a JSON array with exactly {n} objects:\n"
            '[\n  {"customer_message": "...", "agent_response": "...", "intent": "...", "sentiment": "neutral"}\n]\n\n'
            "sentiment must be: positive, neutral, or negative\n"
            "intent should be a short snake_case label e.g. billing_inquiry, account_access, technical_issue\n"
            "Include a realistic mix of sentiments. Return ONLY the JSON array."
        )
    },
    "Text Classification": {
        "description": "Labelled text examples for training text classification models.",
        "columns": ["text", "label", "confidence"],
        "prompt": lambda domain, n: (
            f"Generate {n} text classification examples about: {domain}\n\n"
            f"Return a JSON array with exactly {n} objects:\n"
            '[\n  {"text": "...", "label": "...", "confidence": "high"}\n]\n\n'
            "confidence must be: high, medium, or low\n"
            "Use 3-5 distinct, meaningful labels relevant to the domain.\n"
            "Distribute labels roughly evenly. Return ONLY the JSON array."
        )
    },
    "Instruction-Response (Alpaca)": {
        "description": "Instruction-following pairs in Alpaca format for LLM fine-tuning.",
        "columns": ["instruction", "input", "output"],
        "prompt": lambda domain, n: (
            f"Generate {n} instruction-following examples about: {domain}\n\n"
            f"Return a JSON array with exactly {n} objects in Alpaca format:\n"
            '[\n  {"instruction": "Task description", "input": "Optional context or empty string", "output": "Expected response"}\n]\n\n'
            "Make instructions diverse: explanations, transformations, analysis, code, summarisation.\n"
            "input can be an empty string if not needed. Return ONLY the JSON array."
        )
    },
    "Sentiment Analysis": {
        "description": "Text samples with sentiment labels and brief reasoning.",
        "columns": ["text", "sentiment", "reasoning"],
        "prompt": lambda domain, n: (
            f"Generate {n} sentiment analysis examples about: {domain}\n\n"
            f"Return a JSON array with exactly {n} objects:\n"
            '[\n  {"text": "...", "sentiment": "positive", "reasoning": "Brief explanation"}\n]\n\n'
            "sentiment must be: positive, negative, or neutral\n"
            "Include a realistic mix of all three sentiments. Return ONLY the JSON array."
        )
    }
}


# ── JSON extraction helper ────────────────────────────────────────────────────
def extract_json(text: str) -> list:
    """Robustly extract a JSON array from LLM output."""
    text = text.strip()
    # Strip markdown code fences if present
    match = re.search(r'```(?:json)?\s*([\s\S]*?)```', text)
    if match:
        text = match.group(1).strip()
    # Find array boundaries
    start = text.find('[')
    end = text.rfind(']')
    if start != -1 and end != -1:
        text = text[start:end + 1]
    return json.loads(text)


# ── Generation function ───────────────────────────────────────────────────────
def generate_dataset(dataset_type, domain, model_name, num_examples):
    """Generate synthetic data and return (DataFrame, status_msg, raw_json)."""
    if not domain.strip():
        return None, "Please enter a topic or domain.", ""

    config = DATASET_CONFIGS[dataset_type]
    cfg = MODELS[model_name]
    prompt = config["prompt"](domain.strip(), int(num_examples))

    try:
        response = cfg["client"].chat.completions.create(
            model=cfg["model"],
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": prompt}
            ],
            temperature=0.9,
        )

        raw = response.choices[0].message.content
        data = extract_json(raw)

        # Ensure all expected columns are present in every row
        for item in data:
            for col in config["columns"]:
                if col not in item:
                    item[col] = ""

        df = pd.DataFrame(data, columns=config["columns"])
        status = f"Generated {len(df)} examples using {model_name}"
        raw_json = json.dumps(data, indent=2)
        return df, status, raw_json

    except Exception as e:
        msg = str(e)
        if any(w in msg.lower() for w in ["connect", "refused", "timeout"]):
            msg = f"Could not connect to {model_name}. If using Ollama, run `ollama serve` in a terminal."
        return None, f"Error: {msg}", ""


# ── Download helper ───────────────────────────────────────────────────────────
def download_file(raw_json, dataset_type, fmt):
    """Save generated data to a temp file and return the path for gr.File."""
    if not raw_json:
        return None
    data = json.loads(raw_json)
    if not data:
        return None

    safe_name = re.sub(r'[^a-z0-9]+', '_', dataset_type.lower()).strip('_')

    if fmt == "JSON":
        tmp = tempfile.NamedTemporaryFile(
            mode='w', suffix='.json', delete=False, prefix=f"{safe_name}_"
        )
        json.dump(data, tmp, indent=2)
        tmp.close()
        return tmp.name
    else:
        tmp = tempfile.NamedTemporaryFile(
            mode='w', suffix='.csv', delete=False, newline='', prefix=f"{safe_name}_"
        )
        writer = csv.DictWriter(tmp, fieldnames=data[0].keys())
        writer.writeheader()
        writer.writerows(data)
        tmp.close()
        return tmp.name


# ── Gradio UI ─────────────────────────────────────────────────────────────────
with gr.Blocks(title="Synthetic Data Generator", theme=gr.themes.Soft(primary_hue="green", secondary_hue="emerald")) as ui:

    gr.Markdown(
        "# Synthetic Data Generator\n"
        "Generate diverse AI/ML training datasets using **DeepSeek Chat** or **Llama 3.2 (Ollama)**.\n"
        "Choose a dataset type, enter your topic, pick a model, and download the results."
    )

    # Hidden state to pass raw JSON between generation and download
    raw_json_state = gr.State("")

    # ── Row 1: Dataset type + model ───────────────────────────────────────────
    with gr.Row():
        with gr.Column(scale=2):
            dataset_type_dd = gr.Dropdown(
                choices=list(DATASET_CONFIGS.keys()),
                value="Q&A Pairs",
                label="Dataset Type"
            )
            type_desc = gr.Textbox(
                value=DATASET_CONFIGS["Q&A Pairs"]["description"],
                label="Description",
                interactive=False
            )
        with gr.Column(scale=1):
            model_radio = gr.Radio(
                choices=list(MODELS.keys()),
                value="DeepSeek Chat",
                label="Model"
            )

    # ── Row 2: Topic + count ──────────────────────────────────────────────────
    with gr.Row():
        with gr.Column(scale=3):
            domain_input = gr.Textbox(
                placeholder="e.g. Python programming, customer onboarding, medical diagnosis, e-commerce returns...",
                label="Topic / Domain"
            )
        with gr.Column(scale=1):
            num_slider = gr.Slider(
                minimum=3, maximum=15, value=5, step=1,
                label="Number of Examples"
            )

    # ── Generate button ───────────────────────────────────────────────────────
    generate_btn = gr.Button("Generate Dataset", variant="primary")
    status_box = gr.Textbox(label="Status", interactive=False)

    # ── Output table ──────────────────────────────────────────────────────────
    output_table = gr.DataFrame(label="Generated Dataset", wrap=True)

    # ── Download section ──────────────────────────────────────────────────────
    gr.Markdown("### Download")
    with gr.Row():
        fmt_radio = gr.Radio(
            choices=["JSON", "CSV"],
            value="JSON",
            label="Format"
        )
        download_btn = gr.Button("Download Dataset")
    download_output = gr.File(label="Your File")

    # ── Event: update description when dataset type changes ───────────────────
    def update_description(dt):
        return DATASET_CONFIGS[dt]["description"]

    dataset_type_dd.change(
        update_description,
        inputs=dataset_type_dd,
        outputs=type_desc
    )

    # ── Event: generate ───────────────────────────────────────────────────────
    generate_btn.click(
        generate_dataset,
        inputs=[dataset_type_dd, domain_input, model_radio, num_slider],
        outputs=[output_table, status_box, raw_json_state]
    )

    # ── Event: download ───────────────────────────────────────────────────────
    download_btn.click(
        download_file,
        inputs=[raw_json_state, dataset_type_dd, fmt_radio],
        outputs=download_output
    )

ui.launch(inbrowser=True)
